In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,RobustScaler
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping

# --- GPU Überprüfung und Setup ---
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"NVIDIA GPU erkannt: {len(physical_devices)} GPU(s) verfügbar.")
    try:
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth aktiviert (verhindert Out-of-Memory Fehler).")
    except RuntimeError as e:
        print(e)
else:
    print("Keine GPU erkannt. Training wird auf der CPU ausgeführt.")

df = pd.read_csv('training_data_ready.csv')
df = df.drop([21,43,92,159], axis=0, errors='ignore')
target_cols = ['x', 'y', 'z', 'Gewicht']
x_cols = [col for col in df.columns if col not in target_cols + ['source_file','Objektname','Objektkategorie','OAufnahme','Schwierigkeit']]

X = df[x_cols]
y = df[target_cols]
    # 3. Train-Test-Split (80% Training, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Keine GPU erkannt. Training wird auf der CPU ausgeführt.


In [18]:
#---------------------------- Ausgaben Beste Modelle: -----------------------
#Platz  | Scaler    | Netz-Struktur      | L2     | Dropout | LR     | Val-Loss (MSE) | Val-MAE
#-----------------------------------------------------------------------------------------------
#1    | robust    | [128, 64]          | 0.1    | 0.0     | 0.05   | 314.078888     | 12.3596
#2    | robust    | [512, 256, 128, 64] | 0.5    | 0.0     | 0.05   | 322.648865     | 12.1665
#3    | robust    | [512, 256, 128, 64] | 0.1    | 0.0     | 0.01   | 348.768555     | 12.1428
#4    | robust    | [256, 128, 64]     | 0.1    | 0.0     | 0.01   | 351.149323     | 11.6752
#5    | minmax    | [512, 256, 128, 64] | 0.0    | 0.1     | 0.005  | 368.512878     | 12.4263

def create_neural_network(X_train_data, y_train_data, target_name):
    # 5. Architektur des Neuronalen Netzes definieren
    model = models.Sequential([
        # Eingabeschicht
        layers.Input(shape=(X_train_data.shape[1],)),

        # 1 versteckte Schicht
        #layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01)),

        # 2 versteckte Schicht
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.5)),
        layers.Dropout(0.1),

        # 3 versteckte Schicht
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.1)),
        layers.Dropout(0.1),
        # Ausgabeschicht: 1 Neuron für die jeweilige Zielvariable
        layers.Dense(1)
    ])

    # 6. Kompilieren
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

    # 7. Training
    print(f"Starte Training für {target_name}...")

    # 3. Early Stopping
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=500,
        restore_best_weights=True,
        verbose=0
    )
    history = model.fit(
        X_train_data, y_train_data,
        epochs=2000,
        batch_size=8,
        validation_split=0.1,
        callbacks=[early_stopping],
        verbose=0
    )

    # 5. Besten Validierungsfehler auslesen
    min_val_loss = min(history.history['val_loss'])
    corresponding_mae = history.history['val_mae'][history.history['val_loss'].index(min_val_loss)]
    epochs_trained = len(history.history['val_loss'])

    print(f" -> {target_name} fertig nach {epochs_trained} Epochen. Bester Val-Loss: {min_val_loss:.4f} mit MAE: {corresponding_mae:.4f}")

    return model

In [19]:
models_dict = {}
for target in target_cols:
    models_dict[target] = create_neural_network(X_train_scaled, y_train[target], target)

Starte Training für x...
 -> x fertig nach 1449 Epochen. Bester Val-Loss: 120.9996 mit MAE: 7.9491
Starte Training für y...
 -> y fertig nach 1890 Epochen. Bester Val-Loss: 218.8808 mit MAE: 9.0549
Starte Training für z...
 -> z fertig nach 2000 Epochen. Bester Val-Loss: 29.3280 mit MAE: 3.3583
Starte Training für Gewicht...
 -> Gewicht fertig nach 2000 Epochen. Bester Val-Loss: 41.1543 mit MAE: 2.6983


In [20]:
import datetime
import re
import numpy as np
import pandas as pd
import tensorflow as tf  # Wichtig, um die Learning Rate sauber auszulesen

# --- Vorhersage mit Einzelmodellen ---
predictions_test = {}

for target in target_cols:
    model = models_dict[target]
    preds = model.predict(X_test_scaled).flatten()
    predictions_test[f"{target}_pred"] = preds

y_pred_df = pd.DataFrame(predictions_test, index=y_test.index)

results = y_test.copy()
for target in target_cols:
    results[f"{target}_pred"] = y_pred_df[f"{target}_pred"]
    results[f"abs_error_{target}"] = np.abs(results[f"{target}_pred"] - results[target])
    results[f"mse_{target}"] = (results[f"{target}_pred"] - results[target]) ** 2

results = pd.concat([results, df.loc[y_test.index, ["source_file"]]], axis=1)

# --- Konsolen-Ausgabe ---
print("Auswertung Testdatensatz:")
for target in target_cols:
    print(f"  - MAE {target}: {results[f'abs_error_{target}'].mean():.4f}")
    print(f"  - MSE {target}: {results[f'mse_{target}'].mean():.4f}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
Auswertung Testdatensatz:
  - MAE x: 15.5441
  - MSE x: 396.8511
  - MAE y: 18.1602
  - MSE y: 692.0543
  - MAE z: 5.3595
  - MSE z: 82.2654
  - MAE Gewicht: 6.4251
  - MSE Gewicht: 89.9782


In [21]:
# Zeigt die ersten 20 Zeilen elegant an
format_dict = {}
for target in target_cols:
    format_dict[target] = "{:.2f}"
    format_dict[f"{target}_pred"] = "{:.2f}"
    format_dict[f"abs_error_{target}"] = "{:.2f}"

results.style.format(
    format_dict,
    decimal=",",
).background_gradient(
    subset=[f"abs_error_{t}" for t in target_cols], cmap="Reds"
)

,x,y,z,Gewicht,x_pred,abs_error_x,mse_x,y_pred,abs_error_y,mse_y,z_pred,abs_error_z,mse_z,Gewicht_pred,abs_error_Gewicht,mse_Gewicht,source_file
103,"20,00","87,50","47,50","51,70","54,03","34,03","1158,076906","21,02","66,48","4419,498536","43,53","3,97","15,738020","49,88","1,82","3,319301",Lumi_V_L_L_6.csv
139,"81,39","46,35","20,00","58,10","57,73","23,66","559,786777","43,31","3,04","9,237162","49,49","29,49","869,697773","61,71","3,61","13,009133",Endboss_L_U_S_X_3.csv
80,"47,50","87,50","20,00","51,70","65,42","17,92","321,079031","51,31","36,19","1309,950703","19,71","0,29","0,083897","47,99","3,71","13,773345",Lumi_L_L_8.csv
58,"25,00","50,00","31,75","127,80","37,33","12,33","152,152108","39,96","10,04","100,714621","37,56","5,81","33,716854","134,60","6,80","46,270593",Rechteck_V_L_L_2.csv
100,"20,00","87,50","47,50","51,70","64,67","44,67","1995,797955","52,73","34,77","1208,729847","33,22","14,28","203,779372","53,18","1,48","2,180284",Lumi_V_L_L_3.csv
30,"53,33","91,00","30,00","443,80","85,77","32,44","1052,295469","58,54","32,46","1053,412324","28,28","1,72","2,958568","469,90","26,10","681,426349",3eck_L_L_4.csv
107,"47,50","32,50","17,25","19,30","42,07","5,43","29,521081","42,67","10,17","103,432587","15,91","1,34","1,794757","21,66","2,36","5,549487",Durch_L_U_D_O_4.csv
84,"47,50","87,50","20,00","51,70","50,16","2,66","7,095542","51,33","36,17","1308,451456","19,58","0,42","0,179054","51,29","0,41","0,171940",Lumi_L_L_12.csv
166,"12,50","120,00","17,50","71,60","42,97","30,47","928,445151","39,00","81,00","6560,414167","16,30","1,20","1,449842","72,06","0,46","0,210896",Rechteck_Lang_L_L_3.csv
111,"47,50","32,50","5,75","19,30","70,03","22,53","507,538279","74,37","41,87","1753,502846","7,91","2,16","4,662491","18,66","0,64","0,408202",Durch_L_U_D_U_2.csv


In [22]:
import datetime
import re
import numpy as np
import pandas as pd
import tensorflow as tf  # Wichtig, um die Learning Rate sauber auszulesen

# --- Vorhersage mit Einzelmodellen für Trainingsdaten ---
predictions_train = {}

for target in target_cols:
    model_t = models_dict[target]
    preds = model_t.predict(X_train_scaled).flatten()
    predictions_train[f"{target}_pred"] = preds

y_pred_df = pd.DataFrame(predictions_train, index=y_train.index)

results = y_train.copy()
for target in target_cols:
    results[f"{target}_pred"] = y_pred_df[f"{target}_pred"]
    results[f"abs_error_{target}"] = np.abs(results[f"{target}_pred"] - results[target])
    results[f"mse_{target}"] = (results[f"{target}_pred"] - results[target]) ** 2

results = pd.concat([results, df.loc[y_train.index, ["source_file"]]], axis=1)

# --- NEU: Keras-Parameter dynamisch extrahieren (exemplarisch am ersten Modell) ---

# 1. Zeitstempel erzeugen (Format: JYYMMDD_HHMMSS)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# 2. Modelltyp (wird "Sequential" sein)
rep_model = models_dict[target_cols[0]]
model_name = rep_model.__class__.__name__

try:
    # Netzarchitektur auslesen: Holt die Neuronenanzahl aller Dense-Schichten
    dense_units = [
        str(layer.units) for layer in rep_model.layers if hasattr(layer, "units")
    ]
    arch_str = "-".join(dense_units)  # Wird zu "128-64-3"

    # Optimizer-Name auslesen (z.B. "Adam")
    opt_name = rep_model.optimizer.__class__.__name__

    # Learning Rate auslesen (Keras speichert diese intern oft als Variable/Tensor)
    try:
        lr = float(tf.keras.backend.get_value(rep_model.optimizer.learning_rate))
    except Exception:
        try:
            lr = float(rep_model.optimizer.learning_rate.numpy())
        except Exception:
            lr = "unknown"

    # String für die Parameter zusammensetzen
    param_str = f"Arch_{arch_str}_{opt_name}_lr_{lr}"

except Exception:
    # Sicherer Fallback, falls die Keras-Version eine andere Struktur erzwingt
    param_str = "keras_nn"

# Dateiname zusammenbauen und ungültige Zeichen entfernen
filename = f"Auswertung_train_{model_name}_{param_str}_{timestamp}.csv"
filename = re.sub(r'[\\/*?:"<>| ]', "_", filename)

# Als CSV speichern (inklusive Index)
#results.to_csv(filename, index=True, sep=";", decimal=",")

# --- Konsolen-Ausgabe ---
print("Auswertung Trainingsdatensatz:")
for target in target_cols:
    print(f"  - MAE {target}: {results[f'abs_error_{target}'].mean():.4f}")
    print(f"  - MSE {target}: {results[f'mse_{target}'].mean():.4f}")

print(f"\n[INFO] Datei erfolgreich gespeichert unter: {filename}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Auswertung Trainingsdatensatz:
  - MAE x: 5.8903
  - MSE x: 57.2294
  - MAE y: 4.6381
  - MSE y: 47.5121
  - MAE z: 1.7556
  - MSE z: 5.9135
  - MAE Gewicht: 3.5130
  - MSE Gewicht: 24.9351

[INFO] Datei erfolgreich gespeichert unter: Auswertung_train_Sequential_Arch_128-64-1_Adam_lr_0.009999999776482582_20260617_110313.csv


In [23]:
# Zeigt die ersten 20 Zeilen elegant an
format_dict = {}
for target in target_cols:
    format_dict[target] = "{:.2f}"
    format_dict[f"{target}_pred"] = "{:.2f}"
    format_dict[f"abs_error_{target}"] = "{:.2f}"

results.style.format(
    format_dict,
    decimal=",",
).background_gradient(
    subset=[f"abs_error_{t}" for t in target_cols], cmap="Reds"
)

,x,y,z,Gewicht,x_pred,abs_error_x,mse_x,y_pred,abs_error_y,mse_y,z_pred,abs_error_z,mse_z,Gewicht_pred,abs_error_Gewicht,mse_Gewicht,source_file
161,"120,00","12,50","17,50","71,60","114,83","5,17","26,688033","20,88","8,38","70,192454","16,08","1,42","2,025261","70,59","1,01","1,015105",Rechteck_Lang_L_U_4.csv
2,"50,00","50,00","50,00","185,50","45,51","4,49","20,126866","49,26","0,74","0,548868","50,52","0,52","0,275078","182,74","2,76","7,638641",4eck_g_O_V_u_3.csv
121,"54,50","47,50","20,00","43,90","51,35","3,15","9,892228","42,54","4,96","24,605527","19,80","0,20","0,041131","46,48","2,58","6,651182",6eck_L_U_6.csv
115,"47,50","32,50","5,75","19,30","46,47","1,03","1,058730","31,36","1,14","1,299525","8,88","3,13","9,793683","19,66","0,36","0,126093",Durch_L_U_D_U_6.csv
70,"87,50","47,50","20,00","51,70","73,99","13,51","182,534176","47,06","0,44","0,195669","18,34","1,66","2,755770","51,78","0,08","0,006347",Lumi_L_U_11.csv
136,"64,61","46,35","20,00","58,10","56,51","8,10","65,608482","46,60","0,25","0,061564","21,03","1,03","1,057545","55,45","2,65","7,014694",Endboss_L_U_S_0_3.csv
47,"31,75","50,00","25,00","127,80","37,15","5,40","29,166609","46,92","3,08","9,514536","28,94","3,94","15,505949","137,58","9,78","95,667538",Rechteck_L_L_7.csv
27,"53,33","91,00","30,00","443,80","54,31","0,98","0,955847","77,23","13,77","189,529402","29,07","0,93","0,869850","453,60","9,80","96,006626",3eck_L_L_1.csv
145,"46,35","81,39","20,00","58,10","46,68","0,33","0,110066","78,05","3,34","11,123343","17,98","2,02","4,060246","55,87","2,23","4,973466",Endboss_L_L_S_Y_3.csv
86,"47,50","20,00","87,50","51,70","42,57","4,93","24,319499","20,11","0,11","0,012164","81,93","5,57","30,977150","50,14","1,56","2,434078",Lumi_H_L_U_1.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error

# Definition der 4 Drop-Varianten
drop_variants = {
    "No Drop": [],
    "Variant 1": [21, 43, 92, 159],
    "Variant 2": [92, 99, 31, 169, 150, 43],
    "Variant 3": [17, 21, 43, 53, 62, 92, 129, 150, 159, 166, 169],
    "Variant 4": [15, 17, 21, 30, 35, 43, 53, 62, 67, 72, 73, 75, 91, 92, 129, 139, 142, 150, 159, 166, 169]
}

# Definition der 5 Modell-Konfigurationen aus Zelle 8c0899a8
model_configs = [
    {"name": "Model 1", "scaler": "robust", "layers": [128, 64], "l2": 0.1, "dropout": 0.0, "lr": 0.05},
    {"name": "Model 2", "scaler": "robust", "layers": [512, 256, 128, 64], "l2": 0.5, "dropout": 0.0, "lr": 0.05},
    {"name": "Model 3", "scaler": "robust", "layers": [512, 256, 128, 64], "l2": 0.1, "dropout": 0.0, "lr": 0.01},
    {"name": "Model 4", "scaler": "robust", "layers": [256, 128, 64], "l2": 0.1, "dropout": 0.0, "lr": 0.01},
    {"name": "Model 5", "scaler": "minmax", "layers": [512, 256, 128, 64], "l2": 0.0, "dropout": 0.1, "lr": 0.005}
]

results_list = []
target_cols = ['x', 'y', 'z', 'Gewicht']

for drop_name, drop_indices in drop_variants.items():
    # Daten laden und vorbereiten
    df_full = pd.read_csv('training_data_ready.csv')
    # Nur existierende Indizes droppen
    indices_to_drop = [i for i in drop_indices if i in df_full.index]
    df_exp = df_full.drop(indices_to_drop, axis=0)

    x_cols = [col for col in df_exp.columns if col not in target_cols + ['source_file','Objektname','Objektkategorie','OAufnahme','Schwierigkeit']]

    X = df_exp[x_cols]
    y = df_exp[target_cols]

    # Train-Test-Split mit Validation Split bei Training (80/20 test -> 90/10 val in train)
    X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1, random_state=42)

    for config in model_configs:
        print(f"Starte {drop_name} mit {config['name']}...")

        # Scaler wählen
        if config["scaler"] == "robust":
            scaler = RobustScaler()
        else:
            scaler = MinMaxScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)

        # Für jede Variable ein eigenes Modell trainieren
        for target in target_cols:
            # Modell aufbauen
            model = models.Sequential()
            model.add(layers.Input(shape=(X_train_scaled.shape[1],)))

            for units in config["layers"]:
                model.add(layers.Dense(units, activation='relu', kernel_regularizer=regularizers.l2(config["l2"])))
                if config["dropout"] > 0:
                    model.add(layers.Dropout(config["dropout"]))

            # Ausgabeschicht für ein einziges Target
            model.add(layers.Dense(1))

            optimizer = tf.keras.optimizers.Adam(learning_rate=config["lr"])
            model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

            early_stopping = EarlyStopping(monitor='val_loss', patience=150, restore_best_weights=True, verbose=0)

            model.fit(
                X_train_scaled, y_train[target],
                epochs=1000,
                batch_size=8,
                validation_data=(X_val_scaled, y_val[target]),
                callbacks=[early_stopping],
                verbose=0
            )

            # Evaluation
            preds_train = model.predict(X_train_scaled, verbose=0).flatten()
            preds_val = model.predict(X_val_scaled, verbose=0).flatten()
            preds_test = model.predict(X_test_scaled, verbose=0).flatten()

            mae_train = mean_absolute_error(y_train[target], preds_train)
            mae_val = mean_absolute_error(y_val[target], preds_val)
            mae_test = mean_absolute_error(y_test[target], preds_test)

            results_list.append({
                "Drop_Variant": drop_name,
                "Model_Config": config["name"],
                "Target": target,
                "Train_MAE": mae_train,
                "Val_MAE": mae_val,
                "Test_MAE": mae_test
            })

results_df = pd.DataFrame(results_list)
display(results_df)